# KYC Document and Form Routing

A complete TensorFlow Keras example with **1,000 distinct synthetic document images**. Four classes, 250 images each. Each image represents a separate synthetic document; no augmented copies cross the split.

**Task:** Route incoming KYC documents to the correct processing queue: identity card, address proof, income proof, or application form.

This is image classification, not OCR, signature verification, forgery detection, or legal KYC approval. Generic layouts and fictional details are used. Synthetic test results do not estimate performance on real scans.

## Step 1 — Prepare Google Colab
Open this notebook in Colab and run cells from top to bottom. The image ZIP is downloaded automatically from your supplied Google Drive link. Upload **kyc_labels.csv** when prompted, or upload it in the Colab Files sidebar before running.

The Drive ZIP must be accessible to **Anyone with the link — Viewer** for this direct download. If it is restricted or Drive blocks downloads, the download cell explains the error. No Drive mounting is required.


In [ ]:
%pip install -q gdown pandas numpy matplotlib scikit-learn pillow


### Download and extract the image ZIP
The cell checks that the download is a real ZIP before extracting it. A valid existing ZIP is reused when the cell is rerun.

In [ ]:
import os
import zipfile
import gdown

# Download this use case's ZIP from Google Drive.
zip_path = "/content/kyc_images.zip"
extract_folder = "/content/kyc_download"
os.makedirs(extract_folder, exist_ok=True)

if not os.path.isfile(zip_path) or not zipfile.is_zipfile(zip_path):
    try:
        gdown.download(id="1UnlP3vdu29GEjMJJCGQbPwbOX36ax7Dk", output=zip_path, quiet=False)
    except Exception as error:
        raise RuntimeError(
            "Cannot download the Google Drive ZIP. Check sharing: "
            "Anyone with the link / Viewer. Drive download quotas can also block access."
        ) from error

if not zipfile.is_zipfile(zip_path):
    raise RuntimeError("The downloaded file is not a valid ZIP. Check Drive sharing and download access.")

with zipfile.ZipFile(zip_path, "r") as zip_file:
    # Keep all extracted entries inside this use case's folder.
    for member in zip_file.infolist():
        destination = os.path.abspath(os.path.join(extract_folder, member.filename))
        if os.path.commonpath([os.path.abspath(extract_folder), destination]) != os.path.abspath(extract_folder):
            raise ValueError("Unsafe path in ZIP: " + member.filename)
    zip_file.extractall(extract_folder)

image_count = 0
for folder, subfolders, filenames in os.walk(extract_folder):
    for filename in filenames:
        if filename.lower().endswith((".png", ".jpg", ".jpeg")):
            image_count += 1
print("ZIP extracted to:", extract_folder)
print("Image files found (including any previews):", image_count)


### Upload kyc_labels.csv manually
Select the matching CSV when the upload button appears. If it is already uploaded to `/content`, the cell uses that copy.

In [ ]:
from google.colab import files

csv_path = "/content/kyc_labels.csv"
if not os.path.isfile(csv_path):
    print("Please select kyc_labels.csv from your computer.")
    uploaded = files.upload()
    if "kyc_labels.csv" not in uploaded:
        raise FileNotFoundError("Upload the file named kyc_labels.csv, then rerun this cell.")
    with open(csv_path, "wb") as file:
        file.write(uploaded["kyc_labels.csv"])
print("Labels CSV ready:", csv_path)


## Step 2 — Import libraries and fix the random seed
Keras is accessed through `tensorflow.keras`. A seed makes the split and initialization repeatable, although hardware can still affect results.

## Step 3 — Read the uploaded CSV and locate the extracted images
Read **kyc_labels.csv** from `/content`. The notebook locates the matching image folder inside the ZIP, whether the archive contains `images/...` directly or an extra parent folder. It converts the CSV image paths to absolute paths, so later loading and prediction work without changing directories.

`filename` is the image path; `label` is the target class; `split` assigns train/validation/test data. The manually uploaded CSV is used even if the ZIP includes its own labels file.


## Step 4 — Display sample documents
Look at the image together with its expected class. For cheques, inspect the date box, amount box and signature area. For KYC, inspect the document layout and field arrangement.

## Step 5 — Keep train, validation and test separate
700 training images update the weights. 150 validation images guide early stopping. 150 untouched test images measure the final model. All variants of a real document must stay in one split when adapting this example.

## Step 6 — Load and resize images in batches
Original images are 512 × 768 pixels. Resize to 128 × 192 while preserving their aspect ratio. Each RGB pixel has three channels. `tf.data` reads paths and batches 32 images; labels remain in the same order. Only training data are shuffled. Normalization occurs inside the model.

## Step 7 — Build four convolution and four pooling layers
A convolution filter is a learned 3 × 3 matrix that finds visual patterns. ReLU introduces nonlinearity. `padding="same"` keeps the spatial dimensions during convolution. Each 2 × 2 max pooling layer keeps the strongest response and halves height and width. Filters start from initialized weights and are learned during training.

| Block | Filters | After convolution | After pooling |
|---|---:|---|---|
| 1 | 16 | 128 × 192 × 16 | 64 × 96 × 16 |
| 2 | 32 | 64 × 96 × 32 | 32 × 48 × 32 |
| 3 | 64 | 32 × 48 × 64 | 16 × 24 × 64 |
| 4 | 128 | 16 × 24 × 128 | 8 × 12 × 128 |

Flatten retains spatial positions, which matter for document fields. Dense combines the learned features. Dropout helps reduce overfitting. Softmax returns four class probabilities. Small rotation and contrast augmentation operate only during training; no horizontal flipping is used for documents.

## Step 8 — Compile the model
Adam updates weights using gradients. Sparse categorical cross entropy compares the four probabilities with the integer class ID. Accuracy counts correct predictions. These are mutually exclusive classes; documents missing multiple cheque fields are outside the current label scheme.

## Step 9 — Train the CNN
An epoch is one pass through the training set. A batch is a smaller group processed together. Backpropagation calculates how each weight contributed to the loss, and Adam adjusts it. Early stopping restores the best validation-loss weights; the test set is not used for training decisions. Training can take several minutes on a CPU.

## Step 10 — Plot learning curves
Falling training loss with rising validation loss suggests overfitting. Training and validation accuracy should be interpreted together. No target accuracy is promised for this dataset.

## Step 11 — Evaluate on the held-out test set
Precision asks how often a predicted class is correct. Recall asks how many true examples of that class were found. F1 balances precision and recall. The confusion matrix shows the actual class on rows and predicted class on columns.

## Step 12 — Display images with actual and predicted labels
The reported score is a softmax probability, not calibrated certainty. Check incorrect predictions visually to understand the failure.

## Step 13 — Predict one supplied image
Change `image_path` to your PNG or JPG path. The input image is displayed before its prediction. A 0.70 threshold demonstrates routing low-score results to manual review; it must be validated and calibrated before practical use. A confident score does not detect unfamiliar documents.

## Step 14 — Save and reload the model
The `.keras` model includes preprocessing and learned weights. Save class names in the same order as the output neurons. Reloading does not require retraining.

## Step 15 — Inspect learned feature maps
Feature maps show responses produced by the first convolution layer. They are learned visual features, not extracted text or a guaranteed explanation of a prediction.

## What to try next
Train with real consented document scans from multiple devices and layouts. Split by customer/document and hold out unseen templates. Include blur, shadows, skew and incomplete crops. Add an unknown-document class or out-of-distribution checks. For multiple simultaneous missing fields, use a multilabel output. For reading names, amounts or dates, add OCR with field-level evaluation. Retain human review for actual cheque/KYC decisions.

References: [TensorFlow image classification](https://www.tensorflow.org/tutorials/images/classification) and [TensorFlow CNN tutorial](https://www.tensorflow.org/tutorials/images/cnn).